In [ ]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# Daten laden
df = pd.read_csv("final_df.csv").rename(
    columns={"Unternehmensname Latin alphabet": "firma"}
)


In [ ]:
# Differenz EKQ berechnen
df['dEKQ'] = df.groupby('firma')['EK_Quote'].diff()

In [ ]:
# Nach Firma und Jahr sortieren
df = df.sort_values(['firma', 'Jahr']).copy()

In [ ]:
# Wichtige Variablen Winsorisieren
cols_to_winsor = ['dEKQ', 'ROA', 'Cash_Ratio']

for col in cols_to_winsor:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

In [ ]:
# Verteilung nach Winsorisierung betrachten

vars_to_plot = ['dEKQ', 'ROA', 'Cash_Ratio']

for var in vars_to_plot:
    plt.figure()
    plt.hist(df[var], bins=50)
    plt.title(f'Histogram: {var}')
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
# Simples Modell um zu schauen, ob Zinsschock einen eigenen Einfluss hat
modell_simp = smf.ols(
    formula="""
    EK_Quote ~ Zinsschock
             + Leverage_exante
             + Liquidity_exante
             + ROA
             + Groesse_Log_Bilanzsumme
             + Cash_Ratio
    """,
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["firma"]}
)

print(modell_simp.summary())

In [ ]:
# Simples Modell um zu schauen, ob Zinswende einen eigenen Einfluss hat
modell_simp_zinswende = smf.ols(
    formula="""
    EK_Quote ~ Zinswende_Dummy
             + Leverage_exante
             + Liquidity_exante
             + ROA
             + Groesse_Log_Bilanzsumme
             + Cash_Ratio
    """,
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["firma"]}
)

print(modell_simp_zinswende.summary())

In [ ]:
# Modell mit Niveau der EK_Quote
model_EKQ = smf.ols(
    "EK_Quote ~ Zinsschock + Zinsschock:Leverage_exante + ROA + Groesse_Log_Bilanzsumme",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["firma"]})

print(model_EKQ.summary())

In [ ]:
# Variablen für die Regression mit dEKQ
reg_vars = [
    "dEKQ",
    "EK_Quote",
    "Zinsschock",
    "ABS",
    "Zinswende_Dummy",
    "Leverage_exante",
    "Liquidity_exante",
    "STDebtShare_exante",
    "ROA",
    "Groesse_Log_Bilanzsumme",
    "Cash_Ratio",
    "firma"
]

# Gemeinsamer Datensatz für alle Regressionen mit dEKQ
df_reg = df[reg_vars].dropna().copy()


In [ ]:
# Regression mit Leverage
model_lev = smf.ols(
    "dEKQ ~ Zinsschock + Zinsschock:Leverage_exante + ROA + Groesse_Log_Bilanzsumme + Cash_Ratio",
    data=df_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["firma"]}
)

print(model_lev.summary())

In [ ]:
# Leverage-Modell
beta_z = 0.0270
beta_int = -0.0617

# Range für Leverage
x = np.linspace(0, 1, 100)

# Marginal Effect
effect = beta_z + beta_int * x

plt.plot(x, effect)
plt.axhline(0, linestyle="--")  # Null-Linie
plt.xlabel("Verschuldungsgrad")
plt.ylabel("Effekt des Verschuldungsgrad auf EK-Quote")
plt.title("Marginal Effect des Zinsschocks")

#plt.savefig("Effekt_Leverage2.png", dpi=300)
plt.show()

In [ ]:
# Regression mit Liquidity
model_liq = smf.ols(
    "dEKQ ~ Zinsschock + Zinsschock:Liquidity_exante + ROA + Groesse_Log_Bilanzsumme",
    data=df_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["firma"]}
)

print(model_liq.summary())

In [ ]:
# Liquidity-Modell
beta_z = -0.0112
beta_int = 0.0300

# Range für Leverage
x = np.linspace(0, 1, 100)

# Marginal Effect
effect = beta_z + beta_int * x

plt.plot(x, effect)
plt.axhline(0, linestyle="--")  # Null-Linie
plt.xlabel("Liquidität")
plt.ylabel("Effekt der Liquidität auf EK-Quote")
plt.title("Marginal Effect des Zinsschocks")

#plt.savefig("Effekt_Liquidity2.png", dpi=300)
plt.show()

In [ ]:
# Regression mit STDebtShare
model_STDS = smf.ols(
    "dEKQ ~ Zinsschock + Zinsschock:STDebtShare_exante + ROA + Groesse_Log_Bilanzsumme + Cash_Ratio",
    data=df_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["firma"]}
)

print(model_STDS.summary())

In [ ]:
# STDebtShare-Modell
beta_z = -0.0342
beta_int = 0.0669

# Range für Leverage
x = np.linspace(0, 1, 100)

# Marginal Effect
effect = beta_z + beta_int * x

plt.plot(x, effect)
plt.axhline(0, linestyle="--")  # Null-Linie
plt.xlabel("STDebtShare")
plt.ylabel("Effekt des STDebtShare auf EK-Quote")
plt.title("Marginal Effect des Zinsschocks")

#plt.savefig("Effekt_STDShare2.png", dpi=300)
plt.show()

In [ ]:
# Regression mit Unternehmensgrösse
model_Gr = smf.ols(
    "dEKQ ~ Zinsschock + Zinsschock:Groesse_Log_Bilanzsumme + ROA + Cash_Ratio",
    data=df_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["firma"]}
)

print(model_Gr.summary())

In [ ]:
# Unternehmensgrösse-Modell
beta_z = -0.0052
beta_int = -0.0003

# Range für Leverage
x = np.linspace(0, 1, 100)

# Marginal Effect
effect = beta_z + beta_int * x

plt.plot(x, effect)
plt.axhline(0, linestyle="--")  # Null-Linie
plt.xlabel("Unternehmensgrösse")
plt.ylabel("Effekt der Unternehmensgrösse auf EK-Quote")
plt.title("Marginal Effect des Zinsschocks")

#plt.savefig("Effekt_Grösse2.png", dpi=300)
plt.show()

In [ ]:
# Modell mit finanziellen Interaktionstermen
modell_gemeinsam = smf.ols(
    formula="""
dEKQ ~ Zinsschock
      + Zinsschock:Leverage_exante
      + Zinsschock:Liquidity_exante
      + Zinsschock:STDebtShare_exante
      + ROA
      + Groesse_Log_Bilanzsumme
      + Cash_Ratio
    """,
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["firma"]}
)

print(modell_gemeinsam.summary())

In [ ]:
# Modell mit Absolutem Zinsschock
modell_abs = smf.ols(
    formula="""
    dEKQ ~ ABS
          + ABS:Leverage_exante
          + ROA
          + Groesse_Log_Bilanzsumme
          + Cash_Ratio
    """,
    data=df_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["firma"]}
)

print(modell_abs.summary())

In [ ]:
# Modell mit Zinswende Dummy
modell_dum = smf.ols(
    formula="""
    dEKQ ~ Zinswende_Dummy
          + Zinswende_Dummy:Leverage_exante
          + ROA
          + Groesse_Log_Bilanzsumme
          + Cash_Ratio
    """,
    data=df_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["firma"]}
)

print(modell_dum.summary())